# Movie Analytics - Data Preparation

Load CSVs, clean, upload to MySQL, verify.

## Imports

In [14]:
import pandas as pd
from pathlib import Path
import pymysql
from sqlalchemy import create_engine

## CSV folder path

In [15]:
csv_folder = Path(r'C:/Users/Chandrasekar A/Documents/Python_vs/Movie_Analytics/Movie_Analytics_csv')

## List CSV files

In [3]:
csv_files = sorted(csv_folder.glob('*.csv'))
for file in csv_files:
    print(file.name)

cast.csv
crew.csv
genres.csv
movie_genres.csv
movie_keywords.csv
movies.csv


## Inspect files

In [4]:
for file in csv_files:
    df = pd.read_csv(file)
    print(f"{file.name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

cast.csv: 24,917 rows, 6 columns
crew.csv: 7,297 rows, 6 columns
genres.csv: 19 rows, 2 columns
movie_genres.csv: 6,888 rows, 2 columns
movie_keywords.csv: 41,701 rows, 3 columns
movies.csv: 2,509 rows, 12 columns


## Load movies

In [5]:
movies = pd.read_csv(csv_folder / 'movies.csv')
movies.head()

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [6]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2509 non-null   int64  
 1   title              2509 non-null   str    
 2   original_language  2509 non-null   str    
 3   release_date       2509 non-null   str    
 4   runtime            2509 non-null   int64  
 5   budget             2509 non-null   float64
 6   revenue            2509 non-null   float64
 7   popularity         2509 non-null   float64
 8   vote_average       2509 non-null   float64
 9   vote_count         2509 non-null   int64  
 10  status             2509 non-null   str    
 11  overview           2509 non-null   str    
dtypes: float64(4), int64(3), str(5)
memory usage: 989.6 KB


## Clean movies

In [7]:
movies = movies.drop_duplicates()
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')

numeric_cols = ['runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count']
for col in numeric_cols:
    movies[col] = pd.to_numeric(movies[col], errors='coerce')

movies['title'] = movies['title'].astype(str).str[:200]
movies['original_language'] = movies['original_language'].astype(str).str[:10]
movies['status'] = movies['status'].astype(str).str[:50]
movies['overview'] = movies['overview'].astype(str).str[:1000]

movies['release_year'] = movies['release_date'].dt.year

print(f"Cleaned. {len(movies):,} movies ready.")

Cleaned. 2,503 movies ready.


## Load and clean other files

In [8]:
genres = pd.read_csv(csv_folder / 'genres.csv')
genres = genres.drop_duplicates()

movie_genres = pd.read_csv(csv_folder / 'movie_genres.csv')
movie_genres = movie_genres.drop_duplicates()

cast = pd.read_csv(csv_folder / 'cast.csv')
cast = cast.drop_duplicates()
cast['actor_name'] = cast['actor_name'].astype(str).str[:200]
cast['character_name'] = cast['character_name'].astype(str).str[:200]

crew = pd.read_csv(csv_folder / 'crew.csv')
crew = crew.drop_duplicates()
crew['person_name'] = crew['person_name'].astype(str).str[:200]
crew['job'] = crew['job'].astype(str).str[:100]
crew['department'] = crew['department'].astype(str).str[:100]

keywords = pd.read_csv(csv_folder / 'movie_keywords.csv')
keywords = keywords.drop_duplicates()
keywords['keyword_name'] = keywords['keyword_name'].astype(str).str[:100]

print("All files loaded and cleaned.")

All files loaded and cleaned.


## Connect to MySQL

In [9]:
MYSQL_HOST = 'localhost'
MYSQL_USER = 'root'
MYSQL_PASSWORD = '1234'
DATABASE_NAME = 'Movie analytics'

connection = pymysql.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)

## Create database

In [10]:
cursor = connection.cursor()
cursor.execute(f"DROP DATABASE IF EXISTS `{DATABASE_NAME}`")
cursor.execute(f"CREATE DATABASE `{DATABASE_NAME}`")
cursor.execute(f"USE `{DATABASE_NAME}`")
cursor.close()

print(f"Database '{DATABASE_NAME}' created.")

Database 'Movie analytics' created.


## Upload to MySQL

In [11]:
engine = create_engine(f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{DATABASE_NAME}')

movies.to_sql('movies', engine, if_exists='replace', index=False)
genres.to_sql('genres', engine, if_exists='replace', index=False)
movie_genres.to_sql('movie_genres', engine, if_exists='replace', index=False)
cast.to_sql('cast', engine, if_exists='replace', index=False)
crew.to_sql('crew', engine, if_exists='replace', index=False)
keywords.to_sql('movie_keywords', engine, if_exists='replace', index=False)

print("Data uploaded.")

Data uploaded.


## Verify row counts

In [12]:
connection = pymysql.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=DATABASE_NAME
)

tables = ['movies', 'genres', 'movie_genres', 'cast', 'crew', 'movie_keywords']

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", connection)
    print(f"{table}: {df['count'].iloc[0]:,} rows")

connection.close()

movies: 2,503 rows
genres: 19 rows
movie_genres: 6,878 rows
cast: 24,902 rows
crew: 7,289 rows
movie_keywords: 41,681 rows


C:\Users\Chandrasekar A\AppData\Local\Temp\ipykernel_26420\2353002693.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", connection)
C:\Users\Chandrasekar A\AppData\Local\Temp\ipykernel_26420\2353002693.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", connection)
C:\Users\Chandrasekar A\AppData\Local\Temp\ipykernel_26420\2353002693.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f

## Sample query - top rated movies

In [13]:
connection = pymysql.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=DATABASE_NAME
)

query = """
SELECT title, vote_average, vote_count, release_year
FROM movies
WHERE vote_count > 1000
ORDER BY vote_average DESC
LIMIT 10
"""

top_movies = pd.read_sql(query, connection)
connection.close()

top_movies

C:\Users\Chandrasekar A\AppData\Local\Temp\ipykernel_26420\2569160630.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_movies = pd.read_sql(query, connection)


,title,vote_average,vote_count,release_year
0,The Shawshank Redemption,8.727,30915,1994
1,Michael,8.696,3846,2026
2,The Godfather,8.686,23255,1972
3,Project Hail Mary,8.659,6750,2026
4,Schindler's List,8.570,17680,1993
5,The Godfather Part II,8.570,14116,1974
6,12 Angry Men,8.566,10192,1957
7,The Dark Knight,8.534,36255,2008
8,Spirited Away,8.533,18619,2001
9,Dilwale Dulhania Le Jayenge,8.507,4607,1995
